## Data acquisition pipeline

This notebook completes the full acquisition pipeline for the PlanetScope basemap data used in this project. Starting from the raw construction site dataset, we:

1. **Defined the spatial and temporal extent** of the study area.  
2. **Queried Planet Basemaps** to identify all relevant monthly mosaics.  
3. **Expanded construction sites by month**, aligning them with the mosaic timeline.  
4. **Retrieved intersecting quads** and assigned each site–month record to exactly one quad.  
5. **Built a compact download list** and executed a controlled, resumable download workflow.  
6. **Generated the quad catalog**, consolidating geometry, usage statistics, time references, and file paths for every quad actually required by the project.

Together, these steps establish a reproducible and well-structured dataset that covers the entire temporal and spatial scope of the construction activities being analyzed.

### 1. Load project dependencies and configuration

This cell initializes the environment and imports all required modules for the workflow.  
It sets the project root, loads configuration variables (paths and API key), and brings in all processing functions used throughout the notebook. These include:

- point preprocessing (`process_points`)
- mosaic retrieval and temporal expansion
- quad discovery and assignment
- quad download utilities

With this setup, the notebook gains access to the full processing pipeline and the directory structure defined in the project.

In [1]:
import os
import sys
import pandas as pd
import geopandas as gpd
import json
from pathlib import Path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(os.path.join(project_root, "src"))

# import config variables
from config import DATA_RAW, DATA_PROCESSED, DATA_QUADS, PLANET_API_KEY
from explore_points import process_points
from mosaics import get_mosaics_for_aoi, expand_points_by_month, get_quads_for_points, assign_quads_to_points, extract_quads_to_download, build_quad_index
from download_quads import create_session, ensure_dir, build_quad_output_path, fetch_quad_metadata, download_quad_file, download_quads_for_list

API KEY Successfully loaded


### 2. Define input and output paths

In this step we specify the paths to the main input dataset and the auxiliary outputs used during the preprocessing phase.  
The construction sites are provided as a WGS84 shapefile, and two additional files (GeoPackage and GeoJSON) are reserved for storing the bounding box of the area of interest (AOI).  

These paths are constructed using the project directory configuration to ensure reproducibility and portability across environments.


In [2]:
shp_path = f"{DATA_PROCESSED}/shp/Construction_sites_WGS84.shp"
out_gpkg_path = f"{DATA_PROCESSED}/gpkg/aoi_bbox.gpkg"
out_json_path = f"{DATA_PROCESSED}/gpkg/aoi_bbox.json"

### 3. Extract temporal range and bounding box from construction sites

We process the construction site dataset to obtain three key elements:
- **`min_d` / `max_d`**: the earliest and latest dates associated with the construction records, defining the temporal window for selecting Planet mosaics.
- **`bbox`**: the spatial extent (bounding box) of all construction sites, exported both as a GeoPackage and a GeoJSON.

These values serve as the foundational inputs for querying Planet Basemaps and retrieving only the mosaics that intersect the relevant time and area of interest.


In [4]:
min_d, max_d, bbox = process_points(shp_path, out_gpkg_path, out_json_path)
min_d, max_d, bbox

Saved AOI to /home/student/Documents/jaar/construction-detection/data/processed/gpkg/aoi_bbox.gpkg


(Timestamp('2020-06-01 00:00:00'),
 Timestamp('2025-11-30 00:00:00'),
 [8.275948994286598, 48.69081463561817, 9.315319944733018, 49.49133849492828])

### 4. Query available Planet mosaics intersecting the AOI

Using the AOI bounding box and the temporal window derived from the construction sites, we query the Planet Basemaps API to retrieve all mosaics that match the following criteria:
- intersect the AOI,
- overlap in time with the construction period,
- and follow the naming convention defined by `name_prefix` (e.g., `ps_`).

The resulting reference table (`df_mosaics`) contains the mosaic ID, name, acquisition period, and normalized year–month identifier.  
This table is saved for later reuse and forms the basis for identifying which quads are needed for each construction site.

In [5]:
df_mosaics = get_mosaics_for_aoi(
    bbox,
    min_d,
    max_d,
    PLANET_API_KEY,
    name_prefix="ps_monthly",
    save_path=f"{DATA_PROCESSED}/mosaics_ref.csv"
)
df_mosaics.head(2)

,mosaic_id,name,time_start,time_end,year_month
202,2c61ff49-c7d6-471c-a042-0675d71f5d77,ps_monthly_sen2_normalized_analytic_8b_sr_subs...,2020-05-01,2020-06-01,2020-05
203,dbd62c78-d3b0-4923-a841-8a44c9ae0723,ps_monthly_sen2_normalized_analytic_8b_sr_subs...,2020-06-01,2020-07-01,2020-06


### 5. Expand construction sites into monthly records

The original construction dataset provides start and end dates per site.  
To align these temporal events with the Planet monthly mosaics, each site is expanded into a sequence of monthly records between its start and end dates.

This step:

- Loads the construction sites (`points_gdf`) including their temporal fields.
- Reads the previously extracted mosaic metadata (`mosaics_df`).
- Uses `expand_points_by_month` to generate one row per **site–month** combination, handling ongoing projects and filtering out records without valid dates.
- Produces a normalized table (`points_by_month`) that becomes the temporal backbone for linking sites to mosaics and quads.

This monthly expansion is essential for ensuring that each construction site is paired with the correct Planet mosaic for its corresponding time period.

In [6]:
# points_gdf con order, Start, End, Ongoing, geometry
points_gdf = gpd.read_file(shp_path)

mosaics_df = pd.read_csv(f"{DATA_PROCESSED}/mosaics_ref.csv")

points_by_month = expand_points_by_month(
    points_gdf=points_gdf,
    mosaics_df=mosaics_df,
    id_col="order",
    date_start_col="Start",
    date_end_col="End",
    ongoing_col="Ongoing",
    month_margin=0,
)

points_by_month.head(2)

,order,Start,End,Ongoing,geometry,year_month,mosaic_id
0,145,2021-03-01,2026-03-31,0.0,MULTIPOINT ((9.14698 48.69336)),2021-03,e1056ead-2018-492f-81fe-45de29f02983
1,145,2021-03-01,2026-03-31,0.0,MULTIPOINT ((9.14698 48.69336)),2021-04,1d47a0b7-d0fe-4b50-a56e-05df6346e1f4


### 6. Retrieve quad footprints intersecting construction sites

For each monthly site record, we identify the corresponding Planet quad(s) that spatially intersect the site location.  
Using `get_quads_for_points`, the workflow:

- Queries the appropriate quad tiles for each mosaic,
- Applies a small optional spatial buffer around each point to avoid precision issues,
- Aggregates all unique quads involved in the analysis,
- And stores their footprints and metadata as a GeoDataFrame (`quads_gdf`).

The resulting quad geometries are exported as a GeoPackage for inspection and validation.  
This step establishes the spatial linkage between construction sites and the specific Planet tiles that must be downloaded.

In [7]:
quads_gdf = get_quads_for_points(
    points_by_month=points_by_month,
    api_key=PLANET_API_KEY,
    buffer_deg=0.0005   # optional small buffer (~50m in lon-lat)
)

quads_gdf.to_file(f"{DATA_PROCESSED}/gpkg/points_quads.gpkg", driver="GPKG")
quads_gdf.head()

,mosaic_id,quad_id,geometry
0,e1056ead-2018-492f-81fe-45de29f02983,1071-1348,"POLYGON ((8.4375 49.38237, 8.4375 49.49667, 8...."
1,e1056ead-2018-492f-81fe-45de29f02983,1072-1348,"POLYGON ((8.61328 49.38237, 8.61328 49.49667, ..."
2,e1056ead-2018-492f-81fe-45de29f02983,1073-1348,"POLYGON ((8.78906 49.38237, 8.78906 49.49667, ..."
3,e1056ead-2018-492f-81fe-45de29f02983,1074-1348,"POLYGON ((8.96484 49.38237, 8.96484 49.49667, ..."
4,e1056ead-2018-492f-81fe-45de29f02983,1075-1348,"POLYGON ((9.14062 49.38237, 9.14062 49.49667, ..."


### 7. Assign each site–month record to its corresponding quad

Using the monthly-expanded construction sites and the set of intersecting quads,  
we associate each site–month entry with the specific `quad_id` and `mosaic_id` that cover its location.

This produces a unified table (`point_quad`) linking:
- the site identifier (`fid`),
- the temporal index (`year_month`),
- the Planet mosaic and quad identifiers,
- and the site geometry.

This mapping is fundamental for connecting each construction site directly to the raster data that will be used in subsequent processing steps.

In [10]:
point_quad = assign_quads_to_points(points_by_month, quads_gdf)
out_point_quad = Path(DATA_PROCESSED) / "gpkg/point_quad.gpkg"
point_quad.to_file(out_point_quad, layer="point_quad", driver="GPKG")
point_quad.head(25)

,order,Start,End,Ongoing,year_month,mosaic_id,quad_id,geometry
0,145,2021-03-01,2026-03-31,0.0,2021-03,e1056ead-2018-492f-81fe-45de29f02983,1076-1342,MULTIPOINT ((9.14698 48.69336))
1,178,2021-03-01,2022-10-31,0.0,2021-03,e1056ead-2018-492f-81fe-45de29f02983,1076-1342,MULTIPOINT ((9.3023 48.69382))
2,191,2021-03-01,2023-11-30,0.0,2021-03,e1056ead-2018-492f-81fe-45de29f02983,1075-1343,MULTIPOINT ((9.07058 48.8233))
3,52,2020-08-01,2022-09-01,0.0,2021-03,e1056ead-2018-492f-81fe-45de29f02983,1072-1344,MULTIPOINT ((8.47988 48.92885))
4,159,2021-02-01,2021-05-01,0.0,2021-03,e1056ead-2018-492f-81fe-45de29f02983,1076-1342,MULTIPOINT ((9.26926 48.73964))
5,269,2021-02-01,2022-09-30,0.0,2021-03,e1056ead-2018-492f-81fe-45de29f02983,1072-1342,MULTIPOINT ((8.58295 48.80669))
6,328,2020-08-01,2022-04-30,0.0,2021-03,e1056ead-2018-492f-81fe-45de29f02983,1074-1343,MULTIPOINT ((8.79 48.91209))
7,294,2020-09-01,2021-11-30,0.0,2021-03,e1056ead-2018-492f-81fe-45de29f02983,1073-1344,MULTIPOINT ((8.72943 48.96914))
8,287,2020-11-01,2023-02-28,0.0,2021-03,e1056ead-2018-492f-81fe-45de29f02983,1074-1343,MULTIPOINT ((8.8375 48.83639))
9,281,2020-11-01,2024-06-30,0.0,2021-03,e1056ead-2018-492f-81fe-45de29f02983,1073-1344,MULTIPOINT ((8.64491 48.96763))


### 8. Extract the list of quads to download and build usage statistics

From the site–month–quad assignments, we derive two key tables:

- **`quads_to_download`**:  
  The unique set of `(mosaic_id, quad_id, year_month)` combinations required for the project.  
  This ensures that each quad is downloaded only once, even if it is used by multiple construction sites.

- **`quad_index`**:  
  A summary table indicating how many construction sites fall within each quad (`n_sites`) and the list of their identifiers (`sites`).  
  This provides a clear overview of quad usage and is used later for validation, catalog creation and patch generation.

Together, these tables define the complete set of Planet quads needed for downstream processing.

In [11]:
quads_to_download = extract_quads_to_download(point_quad)
quad_index = build_quad_index(point_quad)
quad_index.head()


quads_to_download.to_csv(f"{DATA_QUADS}/quads_to_download.csv", index=False)
quad_index.to_csv(f"{DATA_QUADS}/quad_index.csv", index=False)

## Downloading data
### 9. Initialize Planet session and test single-quad download

Before launching the full download, we perform a controlled test using a single quad.  
This verifies that authentication, API access, path construction, and file writing all work correctly in the current environment.

Here we:
1. Create an authenticated Planet API session.  
2. Load the list of quads to download from disk.  
3. Select the first quad and inspect its metadata to confirm the correct structure.

This one-quad test ensures the download pipeline is functioning before scaling to multiple quads.


In [12]:
session = create_session(PLANET_API_KEY)
quads=pd.read_csv(f"{DATA_QUADS}/quads_to_download.csv")
row = quads.iloc[852]  # solo el primero
row

mosaic_id     31303f6f-b5c7-4659-bd47-ad234510ce4c
quad_id                                  1075-1342
year_month                                 2025-08
Name: 852, dtype: object

### 10. Retrieve metadata for the selected quad

Using the authenticated session, we query the Planet Basemaps API to obtain the metadata for the selected quad. This response includes spatial information and, crucially, the direct download link found under `meta["_links"]["download"]`.

This step validates API access and confirms that the quad ID, mosaic ID, and request parameters are correct before attempting any file download.

In [13]:
meta = fetch_quad_metadata(session, row["mosaic_id"], row["quad_id"])
meta

{'_links': {'_self': 'https://api.planet.com/basemaps/v1/mosaics/31303f6f-b5c7-4659-bd47-ad234510ce4c/quads/1075-1342?api_key=PLAKfcdf69ec734842d99fca334dd8d87e02',
  'download': 'https://link.planet.com/basemaps/v1/mosaics/31303f6f-b5c7-4659-bd47-ad234510ce4c/quads/1075-1342/full?api_key=PLAKfcdf69ec734842d99fca334dd8d87e02',
  'items': 'https://api.planet.com/basemaps/v1/mosaics/31303f6f-b5c7-4659-bd47-ad234510ce4c/quads/1075-1342/items?api_key=PLAKfcdf69ec734842d99fca334dd8d87e02',
  'thumbnail': 'https://tiles.planet.com/basemaps/v1/planet-tiles/ps_monthly_sen2_normalized_analytic_8b_sr_subscription_2025_08_mosaic/gmap/11/1075/705.png?api_key=PLAKfcdf69ec734842d99fca334dd8d87e02'},
 'bbox': [8.964843748811116,
  48.69096038583766,
  9.140624998786704,
  48.80686345599298],
 'id': '1075-1342',
 'percent_covered': 100}

### 11. Build the local output path for the quad

Based on the quad’s `mosaic_id`, `quad_id`, and `year_month`,  
we generate the expected output path on disk.  
This ensures that each quad is stored in a structured and reproducible directory layout:

data/quads/YYYY_MM/mosaic_id/quad_id.tif


Reviewing the computed path helps verify that the download target is correct  
before writing any raster files.

In [14]:
output_path = build_quad_output_path(row, DATA_QUADS)
output_path

PosixPath('/home/student/Documents/jaar/construction-detection/data/quads/2025_08/31303f6f-b5c7-4659-bd47-ad234510ce4c/1075-1342.tif')

### 12. Download a single quad for validation

With the download URL extracted from the quad metadata, we perform the first end-to-end download test. This step verifies that:

- the quad can be retrieved through the Planet Basemaps API,  
- the output directory is correctly created,  
- and the raster is written to disk without errors.

Running a single download allows us to validate the pipeline before scaling to multiple quads.

In [15]:
download_url = meta["_links"]["download"]
quads=pd.read_csv(f"{DATA_QUADS}/quads_to_download.csv")
result = download_quad_file(
    session=session,
    download_url=download_url,
    output_path=output_path,
    overwrite=False
)
result

{'path': '/home/student/Documents/jaar/construction-detection/data/quads/2025_08/31303f6f-b5c7-4659-bd47-ad234510ce4c/1075-1342.tif',
 'status': 'skipped'}

### 13. Test download of three quads

To further validate the stability of the download workflow, we extend the test to the first three quads in the list. For each quad, we:

1. Request its metadata,  
2. Extract the download URL,  
3. Build the corresponding output path,  
4. Download the raster to disk.

This small batch test confirms that the process works consistently across multiple quads, reducing the risk of unexpected failures during the full download.

In [ ]:
for i in range(3):
    r = quads.iloc[i]
    meta = fetch_quad_metadata(session, r["mosaic_id"], r["quad_id"])
    download_url = meta["_links"]["download"]
    output_path = build_quad_output_path(r, DATA_QUADS)
    
    result = download_quad_file(session, download_url, output_path)
    print(i, result)

### 14. Full quad download

After validating the workflow with one and three quads, we launch the full download process. The function iterates through every required quad in `quads_to_download`, retrieves the corresponding metadata, and stores the raster file in the structured directory hierarchy defined for the project.

A log is returned summarizing the status of each download (`downloaded`, `skipped`, or `failed`).
This provides a complete trace of the operation and ensures the process can be resumed if necessary without re-downloading existing files.

In [16]:
log_full = download_quads_for_list(
    quads_df=quads,
    api_key=PLANET_API_KEY,
    base_dir=DATA_QUADS
)

10/980 processed...
20/980 processed...
30/980 processed...
40/980 processed...
50/980 processed...
60/980 processed...
70/980 processed...
80/980 processed...
90/980 processed...
100/980 processed...
110/980 processed...
120/980 processed...
130/980 processed...
140/980 processed...
150/980 processed...
160/980 processed...
170/980 processed...
180/980 processed...
190/980 processed...
200/980 processed...
210/980 processed...
220/980 processed...
230/980 processed...
240/980 processed...
250/980 processed...
260/980 processed...
270/980 processed...
280/980 processed...
290/980 processed...
300/980 processed...
310/980 processed...
320/980 processed...
330/980 processed...
340/980 processed...
350/980 processed...
360/980 processed...
370/980 processed...
380/980 processed...
390/980 processed...
400/980 processed...
410/980 processed...
420/980 processed...
430/980 processed...
440/980 processed...
450/980 processed...
460/980 processed...
470/980 processed...
480/980 processed...
4

In [17]:
tifs = list(Path(DATA_QUADS).rglob("*.tif"))
len(quads)-len(tifs)

0

### 15. Build the final quad catalog

To consolidate all quad-related information into a single reference dataset, we integrate three elements:

1. **Quad footprints** (`quads_gdf`), providing the spatial geometry of each quad.  
2. **Quad usage statistics** (`quad_index`), indicating how many construction sites fall within each quad and the corresponding site IDs.  
3. **Local file paths**, pointing to the downloaded quad rasters stored on disk.

Only quads that are actually used by at least one construction site are retained. For each quad, we attach its `year_month`, `n_sites`, `sites`, and the full path to the corresponding `.tif` file.

The resulting `quad_catalog` serves as the central index for all downloaded PlanetScope data and is saved both as:

- a **GeoPackage** (`quad_catalog.gpkg`) for spatial inspection in QGIS, and  
- a **CSV** (`quad_catalog.csv`) for downstream scripting and analysis.

This catalog completes the data acquisition phase and provides a clean, structured foundation for patch generation and model training.


In [18]:
quads_gdf= gpd.read_file(f"{DATA_PROCESSED}/gpkg/points_quads.gpkg")
quads_gdf.head()
quad_index= pd.read_csv(f"{DATA_QUADS}/quad_index.csv")                   
quad_index.head()

# 1) Merge quad footprints (quads_gdf) with quad usage info (quad_index)
#    This adds year_month, n_sites, and the list of construction sites per quad.
quads_full = quads_gdf.merge(
    quad_index[["mosaic_id", "quad_id", "year_month", "n_sites", "sites"]],
    on=["mosaic_id", "quad_id"],
    how="left"
)

# 2) Keep only quads that are actually used by at least one construction site
quads_full = quads_full[
    quads_full["n_sites"].notna() & (quads_full["n_sites"] > 0)
].copy()

# 3) Build file paths to the downloaded quad rasters
base_dir = Path(DATA_QUADS)

def build_quad_path(row):
    ym_folder = str(row["year_month"]).replace("-", "_")
    return base_dir / ym_folder / row["mosaic_id"] / f"{row['quad_id']}.tif"

quads_full["path"] = quads_full.apply(build_quad_path, axis=1).astype(str)

# 4) Create the final quad catalog GeoDataFrame
quad_catalog = gpd.GeoDataFrame(
    quads_full[[
        "mosaic_id",
        "quad_id",
        "year_month",
        "n_sites",
        "sites",
        "path",
        "geometry"
    ]],
    geometry="geometry",
    crs=quads_gdf.crs
)

# 5) Save catalog as GeoPackage (for QGIS) and as CSV (for scripting)
out_gpkg = Path(DATA_PROCESSED) / "gpkg/quad_catalog.gpkg"
quad_catalog.to_file(out_gpkg, layer="quad_catalog", driver="GPKG")

out_csv = Path(DATA_PROCESSED) / "quad_catalog.csv"
quad_catalog.drop(columns="geometry").to_csv(out_csv, index=False)

print("quad_catalog saved:")
print(" -", out_gpkg)
print(" -", out_csv)
print(f"Total quads in catalog: {len(quad_catalog)}")



quad_catalog saved:
 - /home/student/Documents/jaar/construction-detection/data/processed/gpkg/quad_catalog.gpkg
 - /home/student/Documents/jaar/construction-detection/data/processed/quad_catalog.csv
Total quads in catalog: 980


### Next Steps

With all PlanetScope data downloaded, indexed, and cataloged, the workflow now moves into:

- **Macro patch generation** around each construction site,  
- **Manual labeling** of construction vs. non-construction areas, and  
- **Micro patch extraction** for training machine-learning models.

These stages will define the training dataset used for segmentation and change-detection
experiments. The quad catalog produced here serves as the foundation for all subsequent
patch-based operations.
